# Tablero de completitud de cuentas contables — DWH

**¿Qué hace este notebook?** Se conecta al Data Warehouse (DWH) y revisa, para el **último año
(12 meses)**, si el campo `VALOR_RESERVA_CONTABLE` (la plata que la aseguradora aparta como
respaldo) tiene valor o está vacío, por cada combinación de **fuente + cuenta contable + libro**.

Es la versión con historia del control `control_completitud_cuentas.sql` (que revisa un solo mes).

**¿Cuándo correrlo?** Los días **8 o 9** de cada mes, antes del proceso oficial de cierre (día 10),
para poder avisar a tiempo si a alguna cuenta le falta información.

**Cómo leer el resultado (semáforo):**
- 🟢 **COMPLETO** = todos los registros de esa cuenta/libro/mes tienen valor → OK para cerrar.
- 🔴 **ALERTA** = hay registros con el valor vacío (nulo) → avisar al dueño de la tabla.

**Cuentas que NO se revisan aquí** (tienen sus propios controles): Incurrido, Salvamentos,
Recobros, IVA AG (se excluye con el filtro `Libro <> 'AG'`) + 2 cuentas por confirmar con Andrey.

> Las credenciales se leen de `credenciales_local.py`, que está en el `.gitignore`
> y **nunca** debe subirse al repositorio.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Credenciales locales (archivo ignorado por git, vive solo en este computador)
from credenciales_local import (
    DEFAULT_HOST_DWH, DEFAULT_PORT_DWH, DEFAULT_DATABASE_DWH,
    DEFAULT_USER_DWH, DEFAULT_PASSWORD_DWH,
)

## 1. Conexión al DWH

Creamos la conexión al servidor SQL Server del Data Warehouse y hacemos una prueba mínima
(`SELECT 1`): si la celda imprime `conexion_ok = 1`, la conexión funciona.

In [2]:
url = URL.create(
    "mssql+pyodbc",
    username=DEFAULT_USER_DWH,
    password=DEFAULT_PASSWORD_DWH,
    host=DEFAULT_HOST_DWH,
    port=int(DEFAULT_PORT_DWH),
    database=DEFAULT_DATABASE_DWH,
    query={"driver": "SQL Server"},  # driver ODBC disponible en esta máquina
)
engine = create_engine(url)

pd.read_sql("SELECT 1 AS conexion_ok", engine)

,conexion_ok
0,1


## 2. ¿Qué meses vamos a revisar?

En vez de escribir el periodo a mano, le preguntamos a las propias tablas cuál es el **último
periodo contable cargado** (formato `AAAAMM`, por ejemplo `202607` = julio de 2026) y desde ahí
contamos **12 meses hacia atrás**. Así el tablero siempre muestra el último año disponible sin
tener que editar nada.

> Solo se consulta el máximo en `DIRECTA` y `CEDIDAS_IAXIS`, que responden en segundos.
> Las otras dos tablas (`CEDIDAS_AS400` y `CEDIDAS_TERREMOTO`) tardan varios minutos incluso
> para un simple `MAX` (no tienen índice por periodo), y comparten el mismo calendario de carga.

In [3]:
sql_ultimo_periodo = """
SELECT MAX(p) AS ultimo_periodo
FROM (
    SELECT MAX(periodo_contable_analisis) AS p FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ
    UNION ALL
    SELECT MAX(periodo_contable_analisis)      FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS
) t
"""
ULTIMO_PERIODO = int(pd.read_sql(sql_ultimo_periodo, engine).iloc[0, 0])


def periodos_hacia_atras(periodo_final: int, n: int = 12) -> list[int]:
    """Lista de n periodos AAAAMM terminando en periodo_final (incluido)."""
    anio, mes = divmod(periodo_final, 100)
    periodos = []
    for _ in range(n):
        periodos.append(anio * 100 + mes)
        mes -= 1
        if mes == 0:
            anio, mes = anio - 1, 12
    return sorted(periodos)


PERIODOS = periodos_hacia_atras(ULTIMO_PERIODO, 12)
print(f"Último periodo cargado en DWH: {ULTIMO_PERIODO}")
print(f"Ventana de análisis (12 meses): {PERIODOS[0]} → {PERIODOS[-1]}")

Último periodo cargado en DWH: 202606
Ventana de análisis (12 meses): 202507 → 202606


## 3. El control de completitud (12 meses en una sola consulta)

Es exactamente la misma lógica de `control_completitud_cuentas.sql` — mismas 4 fuentes, mismas
cuentas, mismos filtros (`Libro <> 'AG'`, etc.) — con un solo cambio: en vez de revisar **un**
periodo, revisa los **12** de la ventana.

Columnas del resultado:
- `total_registros`: cuántas filas hay para esa cuenta/libro/mes.
- `con_valor` / `sin_valor`: cuántas tienen el valor de la reserva y cuántas lo tienen vacío (nulo).
- `pct_completitud`: porcentaje con valor (100 = perfecto).
- `semaforo`: COMPLETO o ALERTA.

> ⏳ **Paciencia**: esta celda puede tardar **varios minutos** — dos de las tablas no tienen
> índice por periodo y el motor tiene que recorrerlas completas. Es normal.

In [4]:
# Los periodos son enteros calculados por nosotros (no texto del usuario),
# por eso es seguro insertarlos directamente en el SQL.
lista_periodos = ", ".join(str(int(p)) for p in PERIODOS)

query_control = f"""
SELECT
    fuente,
    cuenta,
    libro,
    periodo_contable_analisis,
    total_registros,
    con_valor,
    sin_valor,
    CAST(ROUND(100.0 * con_valor / NULLIF(total_registros, 0), 2) AS DECIMAL(5,2)) AS pct_completitud,
    semaforo
FROM
(
    /* 1. DIRECTA — IAXIS */
    SELECT
        'DIRECTA_IAXIS' AS fuente,
        a.CUENTA        AS cuenta,
        a.Libro         AS libro,
        a.periodo_contable_analisis,
        COUNT(*)        AS total_registros,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END) AS con_valor,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END) AS sin_valor,
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END AS semaforo
    FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('410305','410310','410315','510305','510310','510315')
      AND a.Libro <> 'AG'
    GROUP BY a.CUENTA, a.Libro, a.periodo_contable_analisis

    UNION ALL

    /* 2. CEDIDAS — AS400 */
    SELECT
        'CEDIDAS_AS400' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 3. CEDIDAS — IAXIS */
    SELECT
        'CEDIDAS_IAXIS' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 4. CEDIDAS TERREMOTO */
    SELECT
        'CEDIDAS_TERREMOTO' AS fuente,
        a.CUENTA,
        CAST(a.LIBRO AS VARCHAR(20)),
        a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_TERREMOTO_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND UPPER(LTRIM(RTRIM(CAST(a.FUENTE_INTERFAZ AS VARCHAR(50))))) = 'TERR'
      AND ISNULL(LTRIM(RTRIM(CAST(a.LIBRO AS VARCHAR(20)))), '') <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis
) resumen
"""

df = pd.read_sql(query_control, engine)
df["pct_completitud"] = df["pct_completitud"].astype(float)
df["pct_sin_valor"] = (100.0 * df["sin_valor"] / df["total_registros"]).round(2)

print(f"{len(df)} filas (combinaciones fuente + cuenta + libro + mes)")
df.head(10)

240 filas (combinaciones fuente + cuenta + libro + mes)


,fuente,cuenta,libro,periodo_contable_analisis,total_registros,con_valor,sin_valor,pct_completitud,semaforo,pct_sin_valor
0,DIRECTA_IAXIS,410305,AL,202605,7286000,7286000,0,100.0,COMPLETO,0.0
1,DIRECTA_IAXIS,510305,AA,202508,308879,308879,0,100.0,COMPLETO,0.0
2,DIRECTA_IAXIS,510305,AA,202511,508939,508939,0,100.0,COMPLETO,0.0
3,DIRECTA_IAXIS,410310,AL,202508,489857,489857,0,100.0,COMPLETO,0.0
4,DIRECTA_IAXIS,410310,AL,202511,190155,190155,0,100.0,COMPLETO,0.0
5,DIRECTA_IAXIS,410315,AL,202603,90031,90031,0,100.0,COMPLETO,0.0
6,DIRECTA_IAXIS,410315,AL,202606,65536,65536,0,100.0,COMPLETO,0.0
7,DIRECTA_IAXIS,510305,AA,202606,487096,487096,0,100.0,COMPLETO,0.0
8,DIRECTA_IAXIS,510305,AL,202507,9628882,9628882,0,100.0,COMPLETO,0.0
9,DIRECTA_IAXIS,510305,AL,202510,13424853,13424853,0,100.0,COMPLETO,0.0


## 4. Tablero

Colores y estilo compartidos por todos los gráficos (paleta accesible, validada para daltonismo).

In [5]:
# --- paleta (validada; ver skill dataviz / references/palette.md) ---
SURFACE  = "#fcfcfb"   # fondo de los gráficos
INK      = "#0b0b0b"   # texto principal
INK_2    = "#52514e"   # texto secundario
MUTED    = "#898781"   # ejes y etiquetas menores
GRID     = "#e1e0d9"   # líneas de la cuadrícula
FONT     = 'system-ui, -apple-system, "Segoe UI", sans-serif'

# Un color fijo por fuente (nunca cambia aunque se filtre)
COLOR_FUENTE = {
    "DIRECTA_IAXIS":     "#2a78d6",  # azul
    "CEDIDAS_AS400":     "#1baf7a",  # aqua
    "CEDIDAS_IAXIS":     "#eda100",  # amarillo
    "CEDIDAS_TERREMOTO": "#008300",  # verde
}

# Rampa secuencial (un solo tono, claro → oscuro) para el mapa de calor
RAMPA_AZUL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

# Colores de estado (reservados para el semáforo, siempre con ícono + texto)
STATUS_GOOD, STATUS_CRITICAL = "#0ca30c", "#d03b3b"


def layout_base(fig, titulo, alto=420):
    fig.update_layout(
        title=dict(text=titulo, font=dict(color=INK, size=16)),
        font=dict(family=FONT, color=INK_2, size=12),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        height=alto, margin=dict(l=10, r=10, t=50, b=10),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=MUTED), zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=MUTED), zeroline=False)
    return fig

### 4.1 Resumen del último mes (KPIs)

La foto rápida del periodo más reciente: ¿puedo correr el cierre tranquilo o hay que avisar a alguien?

In [6]:
ultimo_con_datos = int(df["periodo_contable_analisis"].max())
d_ult = df[df["periodo_contable_analisis"] == ultimo_con_datos]

pct_global = 100.0 * d_ult["con_valor"].sum() / d_ult["total_registros"].sum()
n_alerta   = int((d_ult["semaforo"] == "ALERTA").sum())
n_completo = int((d_ult["semaforo"] == "COMPLETO").sum())

if n_alerta > 0:
    peor = (d_ult[d_ult["semaforo"] == "ALERTA"].groupby("fuente")["sin_valor"].sum().idxmax())
    estado_icono, estado_color, estado_texto = "⚠", STATUS_CRITICAL, f"{n_alerta} en ALERTA"
else:
    peor = "—"
    estado_icono, estado_color, estado_texto = "✓", STATUS_GOOD, "Sin alertas"

def tile(etiqueta, valor, sub=""):
    return f"""
    <div style="flex:1;min-width:170px;background:{SURFACE};border:1px solid rgba(11,11,11,0.10);
                border-radius:8px;padding:14px 16px;font-family:{FONT};">
      <div style="font-size:12px;color:{MUTED};margin-bottom:4px;">{etiqueta}</div>
      <div style="font-size:26px;color:{INK};font-weight:600;">{valor}</div>
      <div style="font-size:12px;color:{INK_2};margin-top:2px;">{sub}</div>
    </div>"""

display(HTML(f"""
<div style="display:flex;gap:12px;flex-wrap:wrap;">
  {tile("Último periodo con datos", ultimo_con_datos)}
  {tile("Completitud global del mes", f"{pct_global:.2f}%", "con_valor / total_registros")}
  {tile("Estado del semáforo",
        f'<span style="color:{estado_color};">{estado_icono}</span> {estado_texto}',
        f"{n_completo} combinaciones COMPLETO")}
  {tile("Fuente más afectada", peor, "por registros sin valor" if peor != "—" else "")}
</div>"""))

### 4.2 Mapa de calor — historia de 12 meses

**El gráfico principal del tablero.** Cada fila es una combinación fuente + cuenta + libro; cada
columna es un mes. El color muestra el **% de registros SIN valor**:

- Celda **casi blanca / azul muy claro** = 0% sin valor = todo completo (lo normal, lo bueno).
- Celda **azul oscuro** = hay registros vacíos = ahí está el problema.
- Celda **gris vacía (sin color)** = esa cuenta/libro **no tiene ni una fila** en ese mes — también
  es una señal de alerta: puede que la carga de ese mes no haya llegado.

Pasa el mouse por encima de cualquier celda para ver el detalle exacto.

In [7]:
df["serie"] = df["fuente"] + " · " + df["cuenta"].astype(str) + " · " + df["libro"].astype(str).str.strip()

piv_sin  = df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="pct_sin_valor", aggfunc="max")
piv_tot  = df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="total_registros", aggfunc="sum")
piv_nul  = df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="sin_valor", aggfunc="sum")

# Aseguramos las 12 columnas (si un mes no tiene datos, queda como hueco visible)
piv_sin = piv_sin.reindex(columns=PERIODOS)
piv_tot = piv_tot.reindex(columns=PERIODOS)
piv_nul = piv_nul.reindex(columns=PERIODOS)
piv_sin = piv_sin.sort_index(ascending=False)
piv_tot = piv_tot.reindex(piv_sin.index)
piv_nul = piv_nul.reindex(piv_sin.index)

zmax = max(float(piv_sin.max().max() or 0), 5.0)  # escala mínima 5% para que no explote con valores diminutos

hover = []
for s in piv_sin.index:
    fila = []
    for p in PERIODOS:
        v, t, n = piv_sin.loc[s, p], piv_tot.loc[s, p], piv_nul.loc[s, p]
        if pd.isna(v):
            fila.append(f"<b>{s}</b><br>{p}<br>⚠ Sin filas cargadas este mes")
        else:
            fila.append(f"<b>{s}</b><br>{p}<br>Registros: {int(t):,}<br>"
                        f"Sin valor: {int(n):,} ({v:.2f}%)<br>Completitud: {100 - v:.2f}%")
    hover.append(fila)

escala = [[i / (len(RAMPA_AZUL) - 1), c] for i, c in enumerate(RAMPA_AZUL)]

fig = go.Figure(go.Heatmap(
    z=piv_sin.values,
    x=[str(p) for p in PERIODOS],
    y=list(piv_sin.index),
    colorscale=escala, zmin=0, zmax=zmax,
    xgap=2, ygap=2,  # separación de 2px entre celdas
    hoverinfo="text", text=hover,
    colorbar=dict(title=dict(text="% sin valor", font=dict(color=INK_2)),
                  tickfont=dict(color=MUTED), outlinewidth=0),
))
layout_base(fig, "% de registros SIN valor por cuenta/libro y mes (más oscuro = peor)",
            alto=max(380, 32 * len(piv_sin) + 120))
fig.show()

### 4.3 Tendencia — ¿la completitud se mantiene estable?

Porcentaje de completitud de cada **fuente** mes a mes (ponderado: suma de registros con valor
sobre el total de la fuente). Lo esperado es una línea plana pegada al 100%; cualquier caída
es un mes con problemas.

In [8]:
tend = (df.groupby(["fuente", "periodo_contable_analisis"])
          .agg(con=("con_valor", "sum"), tot=("total_registros", "sum"))
          .reset_index())
tend["pct"] = 100.0 * tend["con"] / tend["tot"]

fig = go.Figure()
for fuente, color in COLOR_FUENTE.items():  # orden fijo, el color sigue a la fuente
    d = tend[tend["fuente"] == fuente].sort_values("periodo_contable_analisis")
    if d.empty:
        continue
    fig.add_trace(go.Scatter(
        x=d["periodo_contable_analisis"].astype(str), y=d["pct"],
        name=fuente, mode="lines+markers",
        line=dict(color=color, width=2), marker=dict(size=8, color=color),
        hovertemplate="<b>%{fullData.name}</b><br>%{x}<br>Completitud: %{y:.2f}%<extra></extra>",
    ))
    # etiqueta directa al final de la línea (además de la leyenda)
    fig.add_annotation(x=str(d["periodo_contable_analisis"].iloc[-1]), y=d["pct"].iloc[-1],
                       text=fuente, font=dict(color=INK_2, size=11),
                       xanchor="left", xshift=8, showarrow=False)

layout_base(fig, "Completitud (%) por fuente — últimos 12 meses", alto=420)
fig.update_yaxes(ticksuffix="%")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, font=dict(color=INK_2)),
                  margin=dict(r=140))
fig.show()

### 4.4 Lista de alertas — ¿a quién hay que avisar?

Todas las combinaciones con semáforo **ALERTA** en la ventana de 12 meses, ordenadas por la
cantidad de registros sin valor (las peores primero). Esta es la lista accionable: cada fila
es un aviso pendiente al dueño de la tabla correspondiente.

In [9]:
alertas = (df[df["semaforo"] == "ALERTA"]
           [["fuente", "cuenta", "libro", "periodo_contable_analisis",
             "total_registros", "con_valor", "sin_valor", "pct_completitud"]]
           .sort_values(["sin_valor", "periodo_contable_analisis"], ascending=[False, False])
           .reset_index(drop=True))

if alertas.empty:
    display(HTML(f'<div style="font-family:{FONT};font-size:15px;color:{STATUS_GOOD};">'
                 '✓ <b>Sin alertas</b> — todas las cuentas revisadas están completas '
                 'en los últimos 12 meses.</div>'))
else:
    print(f"⚠ {len(alertas)} alertas en la ventana de 12 meses:")
    display(alertas.style.format({"pct_completitud": "{:.2f}%",
                                  "total_registros": "{:,}",
                                  "con_valor": "{:,}",
                                  "sin_valor": "{:,}"}))

## 5. Validación cruzada contra el script canónico

Para confirmar que este notebook replica bien `control_completitud_cuentas.sql`: los totales de
abajo (último periodo, por fuente) deben coincidir **exactamente** con lo que devuelve ese script
corrido en SSMS con `@PERIODO = {el mismo periodo}`.

In [10]:
(df[df["periodo_contable_analisis"] == ultimo_con_datos]
 .groupby("fuente")[["total_registros", "con_valor", "sin_valor"]]
 .sum())

,total_registros,con_valor,sin_valor
fuente,,,
CEDIDAS_AS400,5893,5893,0
CEDIDAS_IAXIS,104126,104126,0
CEDIDAS_TERREMOTO,181200,181200,0
DIRECTA_IAXIS,21754933,21754933,0


## 6. Cómo interpretar este tablero (guía rápida)

| Si ves… | Significa… | Qué hacer |
|---|---|---|
| KPIs en verde, heatmap claro | Todo completo | OK para el cierre del día 10 |
| Celda oscura en el heatmap | Esa cuenta/libro tiene registros sin valor ese mes | Avisar al dueño de la tabla (ver lista de alertas) |
| Celda vacía (hueco) en el heatmap | No llegó **ninguna** fila ese mes | Revisar si la carga de esa fuente corrió |
| Caída en la línea de tendencia | Un mes malo para toda la fuente | Investigar la carga de ese mes |

**Recordatorios:**
- Este control solo valida que el dato **exista** (no nulo). Validar que el valor "tenga sentido"
  (detectar números atípicos) es la Fase 2, aún no construida.
- Cuentas excluidas por diseño: Incurrido, Salvamentos, Recobros, IVA AG — **+ 2 cuentas
  pendientes de confirmar con Andrey**.
- El control se entrega hasta el semáforo: las alertas automáticas las construye el equipo de
  Andrey por separado (definido en la reunión del 2026-07-03).